# 02 — N-BEATS swap experiment (SARIMA + N-BEATS hybrid)
Clone of Necula et al. 2025 pipeline with **BiLSTM -> N-BEATS**. Same data (`data/transformed_data.csv`, Box-Cox scale), same split (train ≤2022, test 2023), horizons 1/7/30 d. Runtime: Colab GPU kernel (VS Code: select "Colab" kernel) or local CPU.

v0 protocol note: each horizon evaluated as *h-day-ahead forecast from split point* (first h days of 2023). SARIMA order fixed here; align with reproduced baseline (notebook 01) before final paper tables.

In [ ]:
%pip install -q neuralforecast statsmodels

In [ ]:
import time, warnings, itertools
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX

DATA = "../data/transformed_data.csv"
TARGETS = ["PM10", "PM25", "NO2"]
PAPER_BILSTM = {1:(0.1640,0.1468,9.71), 7:(0.1346,0.1150,7.58), 30:(0.0994,0.0792,5.24)}  # RMSE/MAE/MAPE, transformed scale

df = pd.read_csv(DATA); df.columns = [c.strip("\ufeff") for c in df.columns]
df["Date"] = pd.to_datetime(df["Date"], format="%d-%m-%y"); df = df.sort_values("Date").reset_index(drop=True)
df = df.set_index("Date")
train, test = df[df.index < "2023-01-01"], df[df.index >= "2023-01-01"]
print(len(df), "rows | train", len(train), "| test", len(test))

In [ ]:
# Fourier exogenous (annual cycle, k=1..6 -> 12 cols), mirrors paper eq.2
t = np.arange(len(df), dtype=float)
X = np.column_stack([f(2*np.pi*k*t/365) for k in range(1,7) for f in (np.sin,np.cos)])
fourier = pd.DataFrame(X, index=df.index, columns=[f"f{i}" for i in range(X.shape[1])])

In [ ]:
def rmse(y,p): return float(np.sqrt(np.mean((y-p)**2)))
def mae(y,p): return float(np.mean(np.abs(y-p)))
def mape(y,p): return float(np.mean(np.abs((y-p)/np.where(np.abs(y)<1e-8,1e-8,y)))*100)

In [ ]:
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS
import torch
ACC = "gpu" if torch.cuda.is_available() else "cpu"
print("accelerator:", ACC)

SAR = ((2,1,2),(1,1,1,12))  # ponytail: fixed order; replace with AutoARIMA winner from notebook 01 for final protocol

In [ ]:
rows = []
for tgt in TARGETS:
    y = df[tgt]
    for h in (1,7,30):
        # 1) SARIMA on train (+Fourier exog)
        tr_end = len(train)
        m = SARIMAX(y.iloc[:tr_end], exog=fourier.iloc[:tr_end],
                    order=SAR[0], seasonal_order=SAR[1], trend="t").fit(disp=False, method="lbfgs", maxiter=50)
        sar_f = m.get_forecast(h, exog=fourier.iloc[tr_end:tr_end+h]).predicted_mean.values
        actual = y.iloc[tr_end:tr_end+h].values
        t_sar = None
        # 2) residual series on train -> N-BEATS
        resid = (y.iloc[:tr_end] - m.fittedvalues).values
        rd = pd.DataFrame({"unique_id": tgt, "ds": train.index[:len(resid)], "y": resid})
        model = NBEATS(h=h, input_size=60, max_steps=200, enable_progress_bar=False,
                       logger=False, accelerator=ACC,
                       stack_types=("identity",) if h==1 else ("trend","seasonality"))
        nf = NeuralForecast(models=[model], freq="D")
        t0=time.time(); nf.fit(rd); dt=time.time()-t0
        rf = nf.predict()["NBEATS"].values[:h]
        # 3) hybrid = SARIMA + N-BEATS residual correction
        hyb = sar_f + rf
        params = int(sum(p.numel() for p in nf.models[0].parameters()))
        rows.append(dict(target=tgt, h=h, model="SARIMA+NBEATS", rmse=rmse(actual,hyb),
                         mae=mae(actual,hyb), mape=mape(actual,hyb),
                         train_s=round(dt,1), params=params))
        print(rows[-1])

res = pd.DataFrame(rows)
res.to_csv("../results/nbeats_hybrid_v0.csv", index=False)
res

In [ ]:
# Compare vs paper BiLSTM baseline
for h,(br,bm,bp) in PAPER_BILSTM.items():
    sub = res[res.h==h]
    print(f"h={h:>2} | paper BiLSTM RMSE={br:.4f} MAE={bm:.4f} | ours RMSE={sub.rmse.mean():.4f} MAE={sub.mae.mean():.4f} (mean over 3 targets)")

Next steps after v0 run: (1) align SARIMA order with reproduced baseline from notebook 01, (2) rolling-origin eval for wider test coverage, (3) add seeds=3 for mean±std, (4) optional N-HiTS row.